<a href="https://colab.research.google.com/github/elizabethmrankin1/BME450-project/blob/main/BME450_Final_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Breakdown of Dataset 1: Squat Form**

*Train*
- Bad Back = 948 images
- Bad Heel = 852 images
- Good Form = 1001 images

*Test*
- Bad Back = 349 images
- Bad Heel = 321 images
- Good Form = 310 images

Version 1: First Attempt at Training Neural Network

In [ ]:
# =================================================================
# BME 450 Final Project: Squat Form Correction
# This script handles data downloading, preprocessing, and
# baseline model setup for the Bad Back, Bad Heel, and Good classes.
# =================================================================

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
import os

# --- STEP 1: DOWNLOAD AND UNZIP DATA ---
# We use the direct link provided. -O renames it for simplicity.
print("Starting data download...")
!wget "https://zenodo.org/records/17558630/files/Dataset.zip?download=1" -O Dataset.zip

# -q (quiet) hides the 2,800+ file lines. -d specifies the directory.
print("Unzipping files...")
!unzip -q Dataset.zip -d ./squat_data

# --- STEP 2: DATA PREPROCESSING & AUGMENTATION ---
# Per the syllabus and your note on "noisy backgrounds," we use
# transforms to help the model generalize better.
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),      # Resize images for the Neural Network
        transforms.RandomHorizontalFlip(), # Squats look same from left or right
        transforms.RandomRotation(10),     # Account for camera tilt
        transforms.ToTensor(),             # Convert images to mathematical tensors
        # Normalization values standard for models pre-trained on ImageNet
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'test': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# --- STEP 3: LOAD DATA INTO PYTORCH ---
# Point this to the specific folder structure inside the unzipped zip
# Based on your description, the structure is ./squat_data/Dataset/train/
data_dir = './squat_data/Dataset'

# ImageFolder automatically uses subfolder names as labels (0, 1, 2)
image_datasets = {
    'train': datasets.ImageFolder(os.path.join(data_dir, 'train'), data_transforms['train']),
    'test': datasets.ImageFolder(os.path.join(data_dir, 'test'), data_transforms['test'])
}

# DataLoaders handle feeding images to the GPU in batches (32 at a time)
dataloaders = {
    'train': torch.utils.data.DataLoader(image_datasets['train'], batch_size=32, shuffle=True),
    'test': torch.utils.data.DataLoader(image_datasets['test'], batch_size=32, shuffle=False)
}

# Verify the classes found
class_names = image_datasets['train'].classes
print(f"Detected Classes: {class_names}")
print(f"Total training images: {len(image_datasets['train'])}")

# --- STEP 4: DEFINE ARCHITECTURE (Requirement: 1-2 different architectures) ---
# For a strong start, we use Transfer Learning with ResNet18.
# This model is pre-trained on millions of images and is great at ignoring noise.
model = models.resnet18(pretrained=True)

# We must change the "Final Layer" to match our 3 classes: Bad Back, Bad Heel, Good
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 3) # Input features from ResNet -> 3 outputs

# Send the model to the GPU (standard in Colab)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# --- STEP 5: DEFINE LOSS AND OPTIMIZER (Requirement: 4-5 parameter choices) ---
# CrossEntropyLoss is standard for 3-class classification
criterion = nn.CrossEntropyLoss()

# You can experiment with this 'lr' (learning rate) as one of your 5 parameter choices!
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Model setup complete. Ready for training!")

Starting data download...
--2026-04-16 19:45:20--  https://zenodo.org/records/17558630/files/Dataset.zip?download=1
Resolving zenodo.org (zenodo.org)... 188.184.98.114, 137.138.153.219, 188.185.48.75, ...
Connecting to zenodo.org (zenodo.org)|188.184.98.114|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 824064094 (786M) [application/octet-stream]
Saving to: ‘Dataset.zip’

Dataset.zip         100%[===================>] 785.89M  19.6MB/s    in 32s     

2026-04-16 19:45:53 (24.5 MB/s) - ‘Dataset.zip’ saved [824064094/824064094]

Unzipping files...
